[Spark ES instance](https://www.elastic.co/guide/en/elasticsearch/hadoop/master/spark.html) or [Start ES instance in docker](https://www.elastic.co/guide/en/elasticsearch/reference/current/docker.html)

```
DELETE policies
```

```
PUT policies
{
  "settings": {
    "index": {
      "number_of_shards": "8",
      "number_of_replicas": "0",
      "refresh_interval": "-1"
    }
  },
  "mappings": {
    "properties": {
      "Shape": {
        "type": "geo_point"
      },
      "TotalReplacementValue": {
        "type": "double"
      },
      "TotalInsuredValue": {
        "type": "double"
      },
      "Quarter": {
        "type": "keyword"
      },
      "ISOType": {
        "type": "keyword"
      },
      "ConstructionType": {
        "type": "keyword"
      },
      "LineOfBusiness": {
        "type": "keyword"
      },
      "YearBuilt": {
        "type": "keyword"
      },
      "eqID": {
        "type": "integer"
      },
      "eqCode": {
        "type": "double"
      },
      "eqValue": {
        "type": "double"
      },
      "hex100": {
        "type": "keyword"
      },
      "hex200": {
        "type": "keyword"
      },
      "hex1k": {
        "type": "keyword"
      }
    }
  }
}
```

In [ ]:
_ = sql("select 'Initialize Spark Context'").collect()

In [ ]:
esri = spark._jvm.com.esri.spark.Functions
esri.registerFunctions()
esri.registerHex("hex100", 100.)
esri.registerHex("hex200", 200.)
esri.registerHex("hex1k", 1000.)

In [ ]:
gdb_path = "path/to/Insurance.gdb"

In [ ]:
dfPolicy = spark.\
    read.\
    format("com.esri.gdb").\
    option("path", gdb_path).\
    option("name", "POLICY25M").\
    option("numPartitions", 12).\
    load().\
    drop("OBJECTID","Shape")
dfPolicy.createOrReplaceTempView("POLICY25M")
dfPolicy.printSchema()

In [ ]:
sql("""select *,
lon2x(LON) as X,
lat2y(LAT) as Y
from POLICY25M
""").registerTempTable("T1")

In [ ]:
%%time

sql("""select
Quarter,
LineOfBusiness,
TotalInsuredValue,
TotalReplacementValue,
ConstructionType,
ISOType,
YearBuilt,
hex100(X,Y) as hex100,
hex200(X,Y) as hex200,
hex1k(X,Y) as hex1k,
array(LON,LAT) as Shape
from T1
""").\
    write.\
    format("org.elasticsearch.spark.sql").\
    option("es.nodes", "localhost").\
    save("policies")

```
GET policies/_count
```

```
POST _sql?format=txt
{
  "query":"select eqCode,count(TotalInsuredValue) as c,sum(TotalInsuredValue) as s from policies where eqID>-1 group by eqCode order by eqCode"
}
```

```
POST _sql?format=txt
{
  "query":"select YearBuilt,ISOType,count(ISOType) as pop from policies where YearBuilt>='2001' and YearBuilt!='None' group by YearBuilt,ISOType"
}
```

```
POST policies/_search?size=0
{
    "aggs" : {
        "hex1k_count" : {
            "cardinality" : {
                "field" : "hex1k"
            }
        }
    }
}
```